# GAP Experiment: W=96 D=6 BN Skip (TPU, 50 epochs)

**Research question:** Does J_topo's predictive power generalize to skip=True?

SynFlow search space covers only skip=False.
W=96 D=6 BN Skip is the ONE HBO top-5 architecture NOT covered by SynFlow.

HBO result (50 epochs): val_loss=0.458, test_acc=0.844

This run: verify whether the same config achieves similar performance.

In [ ]:
import sys
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

# TPU setup
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

DEVICE = xm.xla_device()
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Data loading — same protocol as HBO_revised notebook
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/tmp/data', train=True, download=True, transform=transform_train)
test_ds = torchvision.datasets.CIFAR10('/tmp/data', train=False, download=True, transform=transform_test)

# Split into train/val (same seed as HBO_revised)
np.random.seed(42)
indices = np.random.permutation(len(full_train))
val_size = 5000
val_idx, train_idx = indices[:val_size], indices[val_size:]
train_ds = Subset(full_train, train_idx)
val_ds = Subset(full_train, val_idx)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# SynFlow search space only covers skip=False.
# The ONE HBO top-5 architecture NOT in SynFlow's coverage: W=96 D=6 BN Skip.
# This tests whether J_topo's predictive power generalizes to skip=True architectures.

GAP_EXPERIMENT = [
    {'width': 96, 'depth': 6, 'norm': 'batchnorm', 'skip': True},   # HBO rank 4  val_loss=0.458
]

EPOCHS_L2 = 50   # L2 final evaluation (skip L1 — just 1 config)
LR = 0.05
SEEDS = [42, 123]
LOG_EVERY = 10

In [ ]:
def build_model(width, depth, norm_type, skip=False):
    """Plain ConvNet matching HBO_revised architecture definition."""
    layers = []
    c = 3
    for i in range(depth):
        layers.append(nn.Conv2d(c, width, 3, padding=1))
        if norm_type == 'batchnorm':
            layers.append(nn.BatchNorm2d(width))
        elif norm_type == 'layernorm':
            layers.append(nn.LayerNorm([width, 32, 32]))
        c = width
    if skip:
        layers.append(nn.Identity())  # placeholder for skip
    layers.extend([
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(c, 10)
    ])
    return nn.Sequential(*layers)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        xm.optimizer_step(optimizer)
        total_loss += loss.item() * x.size(0)
    scheduler.step()
    return total_loss / len(loader.dataset)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * x.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

In [ ]:
def train_model(cfg, epochs, seed=42, log_every=5):
    """Train one config, return full epoch-by-epoch history."""
    torch.manual_seed(seed)
    model = build_model(cfg['width'], cfg['depth'], cfg['norm'], cfg['skip'])
    model = model.to(DEVICE)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = []
    best_val_loss = float('inf')
    best_val_acc = 0.0
    start = time.time()
    
    for ep in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_acc = evaluate(model, val_loader)
        
        # Track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
        
        # Log every N epochs
        if (ep + 1) % log_every == 0 or ep == 0:
            elapsed = time.time() - start
            print(f"    ep={ep+1:3d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}  ({elapsed:.0f}s)", flush=True)
        
        # Checkpoint
        if (ep + 1) % 10 == 0:
            xm.save(model.state_dict(), f"/tmp/ckpt_W{cfg['width']}_D{cfg['depth']}_ep{ep+1}_seed{seed}.pt")
        
        history.append({'epoch': ep+1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})
    
    test_loss, test_acc = evaluate(model, test_loader)
    
    return {
        'history': history,
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'test_loss': test_loss,
        'test_acc': test_acc
    }

In [ ]:
# ── L2: 50 epochs — W=96 D=6 BN Skip ──────────────────────────────────
# This is the one HBO top-5 architecture NOT in SynFlow's search space.
# HBO result (50 epochs): val_loss=0.458, test_acc=0.844

print("=" * 70)
print("L2 (50 epochs) — GAP experiment: W=96 D=6 BN Skip")
print("=" * 70)

cfg = GAP_EXPERIMENT[0]
cfg_label = f"W={cfg['width']} D={cfg['depth']} {cfg['norm']} skip={cfg['skip']}"
print(f"\nConfig: {cfg_label}")

l2_losses = []
l2_accs = []
for seed in SEEDS:
    t0 = time.time()
    result = train_model(cfg, EPOCHS_L2, seed, log_every=10)
    elapsed = time.time() - t0
    l2_losses.append(result['best_val_loss'])
    l2_accs.append(result['test_acc'])
    print(f"  seed={seed}: L2_val={result['best_val_loss']:.4f}  test_acc={result['test_acc']:.4f}  ({elapsed:.0f}s)", flush=True)

l2_mean = np.mean(l2_losses)
l2_std = np.std(l2_losses)
l2_acc_mean = np.mean(l2_accs)
print(f"\n→ L2 mean: {l2_mean:.4f} ± {l2_std:.4f}  test_acc: {l2_acc_mean:.4f}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL RESULTS: GAP experiment — W=96 D=6 BN Skip (50 epochs)")
print("=" * 70)
print(f"\nConfig: {cfg_label}")
print(f"L2 val loss: {l2_mean:.4f} ± {l2_std:.4f}")
print(f"Test acc:    {l2_acc_mean:.4f}")

print("\n" + "-" * 70)
print("Comparison (all 50 epochs, CIFAR-10):")
print("-" * 70)
print(f"{'Method':<20} {'Config':<25} {'Val Loss':>10} {'Test Acc':>10}")
print("-" * 70)
print(f"{'This run (Skip)':<20} {'W=96 D=6 BN Skip':<25} {l2_mean:>10.4f} {l2_acc_mean:>10.4f}")
print(f"{'HBO (NoSkip)':<20} {'W=96 D=6 BN NoSkip':<25} {0.3770:>10.4f} {0.8744:>10.4f}")
print(f"{'Random (NoSkip)':<20} {'W=64 D=6 BN NoSkip':<25} {0.4270:>10.4f} {0.8515:>10.4f}")

delta_vs_hbo = l2_mean - 0.3770
print(f"\nΔ vs HBO (NoSkip): {delta_vs_hbo:+.4f}")

In [ ]:
# Save results
output = {
    'experiment': 'gap_experiment_skip_tpu',
    'epochs_l2': EPOCHS_L2,
    'seeds': SEEDS,
    'config': {
        'width': cfg['width'],
        'depth': cfg['depth'],
        'norm': cfg['norm'],
        'skip': cfg['skip']
    },
    'l2_val_loss': l2_mean,
    'l2_val_loss_std': l2_std,
    'test_acc': l2_acc_mean,
    'hbo_reference_skip': {
        'val_loss': 0.4582,
        'test_acc': 0.8440,
        'config': 'W=96 D=6 BN Skip'
    },
    'hbo_reference_noskip': {
        'val_loss': 0.3770,
        'test_acc': 0.8744,
        'config': 'W=96 D=6 BN NoSkip'
    }
}

with open('/tmp/synflow_l2_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved to /tmp/synflow_l2_results.json")
print("\nJSON:\n")
print(json.dumps(output, indent=2))